# Supplementary figures — rating-prediction block (Figs. S24–S33)

The prediction half of the supplementary information: the rating model, its diagnostics, and
the borrowing-cost panels. Figs. S1–S23 are in `SI_figures1.ipynb`.

| Figure | File | Source |
|---|---|---|
| **S24** | `rated_countries_overtime.png` | **R** — `rated_countries_overtime.R`, in this folder |
| **S25** | `rating_coupons.png` | `2_code/supplementary/figures/rating_coupons.py` |
| **S26** | `rating_coupons_currency_v2.png` | two panels, built here from `prep.plot_bond_coupon_vs_rating` |
| **S27** | `XGBoost_accuracy.png` | `XGB_supplementary_plots.py --s27` |
| **S28** | `XGBclassification_features.png` | `XGB_supplementary_plots.py` |
| **S29** | `XGBoost_heldout_test.png` | `XGB_supplementary_plots.py --s29` |
| **S30** | `decisiontree.png` | `XGB_supplementary_plots.py --s30` |
| **S31** | `M5_rating_notch_full_500seeds.png` | `M5_basispoint_plot.py --metric notches` |
| **S32** | `supp-predictionmodels_a/b.png` | `XGB_supplementary_plots.py --s32` |
| **S33** | `M5_currency.png` | `M5_basispoint_plot.py --metric bps` (run by the main-figure notebook) |

No counterfactual construction here, so this notebook runs in well under a minute plus whatever
the generator scripts take.

## 1. Setup

In [ ]:
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.ion()

# Portable repo-root resolution: walk up to the .git / .here marker, then run from there.
ROOT = os.getcwd()
while not (os.path.isdir(os.path.join(ROOT, '.git')) or os.path.isfile(os.path.join(ROOT, '.here'))) and ROOT != os.path.dirname(ROOT):
    ROOT = os.path.dirname(ROOT)
os.chdir(ROOT)
sys.path.insert(0, os.path.join(ROOT, '2_code', 'common'))

import subprocess
def run_script(path, *args, cwd=None):
    """Run one of the standalone generators and show the tail of its output."""
    r = subprocess.run([sys.executable, path, *args], cwd=cwd, capture_output=True, text=True)
    print((r.stdout or r.stderr)[-700:])
    if r.returncode != 0:
        print(f'   [exit {r.returncode}] {path}')
    return r.returncode

print('project root:', os.path.basename(ROOT))

## 2. Figures

> **Fig. S24** (`rated_countries_overtime.png`) is produced by `rated_countries_overtime.R` in this folder — run it with `Rscript`.

### Fig. S25 — `rating_coupons.png`

In [ ]:
run_script('2_code/supplementary/figures/rating_coupons.py')

### Fig. S26 — `rating_coupons_currency_v2.png`

Two panels: **a** credit rating vs bond coupon rates by currency, all maturities (the published figure); **b** the same restricted to bonds of 10-year maturity

In [ ]:
# Bond-coupon data comes from the rating-model prep module.
import sys as _sys, os as _os
_sys.path.insert(0, _os.path.join(ROOT, '2_code', 'common'))
from prep import load_and_process_all, plot_bond_coupon_vs_rating

_OUT = _os.path.join('4_figures', 'supplementary')
_PANELS = _os.path.join(_OUT, 'rating_coupons_currency_panels')

# Fig. S26 plots individual bonds, so it needs the proprietary Bloomberg/LSEG data (see 2_code/data_prep/DATA.md).
# Without it the cell reports that and the notebook carries on with the other figures.
try:
    _res = load_and_process_all()
except FileNotFoundError as _e:
    _res = None
    print(f'Fig. S26 skipped: {_e}')

if _res is not None:
    _data = _res['processed_data']
    # JPY bonds excluded: their coupons are held artificially low by the BoJ.
    _data_adj = _data[~_data['currency_issued'].isin(['JPY'])]

    _COUPON_KW = dict(score_column='score_total',
                      liquid_currencies=['USD', 'GBP', 'EUR', 'CHF', 'CAD'],
                      max_coupon=30, figsize=(10, 6), show_usd_fit=False)

    _os.makedirs(_PANELS, exist_ok=True)

    # Panel a - all maturities (the published figure).
    plot_bond_coupon_vs_rating(_data_adj, 2011, 2019, show_stats=True,
                               title_suffix='all maturities', panel_label='a',
                               save_path=_os.path.join(_PANELS, 'panel_a_all_maturities.png'),
                               **_COUPON_KW)

    # Panel b - 10-year and shorter; each quadratic fit stops at the last rating with bonds in that currency group.
    plot_bond_coupon_vs_rating(_data_adj, 2011, 2019, show_stats=False,
                               maturity=10, maturity_op='<=', clip_fit_to_data=True,
                               panel_label='b',
                               save_path=_os.path.join(_PANELS, 'panel_b_maturity_le10.png'),
                               **_COUPON_KW)

**Compose panels a and b**

In [ ]:
# Stack the two panels into the submitted figure (only when Fig. S26 was redrawn above).
if _res is not None:
    from PIL import Image
    _a = Image.open(_os.path.join(_PANELS, 'panel_a_all_maturities.png'))
    _b = Image.open(_os.path.join(_PANELS, 'panel_b_maturity_le10.png'))
    _w = max(_a.width, _b.width)
    def _pad(im):
        if im.width == _w:
            return im
        out = Image.new('RGB', (_w, im.height), 'white'); out.paste(im, (0, 0)); return out
    _a, _b = _pad(_a.convert('RGB')), _pad(_b.convert('RGB'))
    _fig = Image.new('RGB', (_w, _a.height + _b.height), 'white')
    _fig.paste(_a, (0, 0)); _fig.paste(_b, (0, _a.height))
    _fig.save(_os.path.join(_OUT, 'rating_coupons_currency_v2.png'))
    print('wrote', _os.path.join(_OUT, 'rating_coupons_currency_v2.png'), _fig.size)

### Figs. S27, S28, S29, S30, S32 — XGB diagnostics

One call into `2_code/supplementary/figures/XGB_supplementary_plots.py`, which holds every XGB figure
generator in this package:

| flag | figure | output |
|---|---|---|
| (default) | **S28** feature importance | `feature_importance_XGB_*.png` |
| `--s27` | **S27** confusion matrices (seed 8, 20-class scale) | `XGBoost_accuracy.png` |
| `--s29` | **S29** accuracy across 50 seeds, 20% country holdout | `XGBoost_heldout_test.png` |
| `--s30` | **S30** one decision tree from the base model | `decisiontree.png` |
| `--s32` | **S32** both panels | `supp-predictionmodels_a.png`, `supp-predictionmodels_b.png` |

**S32 outputs.** Two manuscript files, matching the two `\includegraphics` in the SI:

- `supp-predictionmodels_a.png` — one figure: 4 rating sources x 3 countries.
- `supp-predictionmodels_b.png` — one figure: the three model rows stacked. Each row carries all
  three of its elements (speculative-grade histogram, upgrade histogram, per-country rating PDFs),
  so the rightmost column is *part of* the row, not a separate output.

**Panel-a axes.** All four columns share one scale: the 19 rating classes **CCC- to AAA** on x and
a fixed **0–0.3** probability axis on y. The average-rating model starts at CCC (the 18 classes behind
the headline M5 results), so its CCC- bin is empty.

In [ ]:
run_script('XGB_supplementary_plots.py', '--seeds', '500',
           '--s27', '--s29', '--s30', '--s32',
           cwd='2_code/supplementary/figures')

### Figs. S31, S33 — rating-notch and currency panels

`--metric notches` writes Fig. S31; the `bps` run in the main-figure notebook writes Fig. S33 (`M5_bp_currency`).

In [ ]:
run_script('M5_basispoint_plot.py', '--seeds', '500', '--all-isos',
           '--metric', 'notches', cwd='2_code/main/figures')

## 4. Model runs these figures depend on

Figs. S27, S29 and S32 read prediction/metric files that `XGB_model.py` writes:

```bash
cd 2_code/main/analysis/rating_model
# S27, S28, S32 row 1 (and Fig. 5): the 500-seed multi-scenario run
python3 generate_XGB_predictions.py --seeds 500
# S29 - both panels, on the S&P label (see below)
python3 XGB_model.py --model XGB_classification --scoring neglogloss --seeds 0 50 --rating-source sp
python3 XGB_model.py --model XGB_classification --scoring neglogloss --seeds 0 50 --rating-source sp --restrict-features
# S32 panel b, rows 2 and 3
python3 XGB_model.py --model XGB_classification --scoring accuracy   --seeds 0 10
python3 XGB_model.py --model XGBO               --scoring neglogloss --seeds 0 10
# S32 panel a - one short run per rating source
for src in sp fitch moodys total; do
  python3 XGB_model.py --model XGB_classification --scoring neglogloss --seeds 0 10 --rating-source $src
done
```

**`--restrict-features`** drops rows with any missing feature — Fig. S29's "Restricted sample".
The default keeps them (XGBoost handles NaN natively), which is its "Full sample".

**`--rating-source`** picks the label: `total` (default, the across-agency average used for every
headline result) or `sp` / `fitch` / `moodys`. Fig. S29 uses **`sp`** (its Full sample, n=3128, is the S&P sample).

The four panel-a runs share one prediction cross-section: every country rated by *any* agency in
2019 (predictions need features, not a label from that agency).

A country holdout can leave a rare rating class out of a training split, which XGBoost rejects;
those seeds are skipped and reported rather than aborting the run.